In [ ]:
import pandas as pd
import scanpy as sc
import anndata as ad
from pathlib import Path

figdir = Path("/gpfs/Home/esm5360/bulk_rna_seq_analysis/figures/Step_030_merge_wt_and_ko")
figdir.mkdir(parents=True, exist_ok=True)
sc.settings.figdir = str(figdir)

## Merging WT and KO

In [ ]:
wt_filtered_adata = sc.read_h5ad("/gpfs/Home/esm5360/bulk_rna_seq_analysis/data/wt_filtered_adata.h5ad")
ko_filtered_adata = sc.read_h5ad("/gpfs/Home/esm5360/bulk_rna_seq_analysis/data/ko_filtered_adata.h5ad")

In [ ]:
common_genes = ko_filtered_adata.var_names.intersection(wt_filtered_adata.var_names)

ko = ko_filtered_adata[:, common_genes].copy()
wt = wt_filtered_adata[:, common_genes].copy()

ko.obs['condition'] = 'KO_Sample01'
wt.obs['condition'] = 'WT_Sample01'

adata = sc.concat([ko, wt], label='batch', keys=['KO_Sample01', 'WT_Sample01'])

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
# Find clusters (Seurat FindClusters)
sc.tl.leiden(adata, resolution=0.05, flavor="igraph") 

# Visualize
sc.pl.umap(adata, color=['condition'], groups=['KO_Sample01', 'WT_Sample01'])

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
colors = {'KO_Sample01': '#f3746d', 'WT_Sample01': '#19bdc2'}

# Plot UMAP colored by condition
sc.pl.umap(adata, color='condition', show=False, palette=colors, size=5, ax=ax)

# Get UMAP coords
coords = adata.obsm['X_umap']
df = pd.DataFrame(coords, columns=['UMAP1', 'UMAP2'])
df['leiden'] = adata.obs['leiden'].values

# Compute cluster centroids
centroids = df.groupby('leiden')[['UMAP1', 'UMAP2']].mean()

# Label clusters
for cluster, (x, y) in centroids.iterrows():
    plt.text(x, y, cluster, fontsize=14, weight='bold')
    
plt.legend(
    bbox_to_anchor=(1.0, 0.5), loc='center left', borderaxespad=0.,
    frameon=False, fontsize=14
)
plt.xlabel('UMAP1', fontsize=14)
plt.ylabel('UMAP2', fontsize=14)
plt.title('', fontsize=16)
plt.tight_layout()
fig.savefig(figdir / "combined_umap.png", dpi=300)
plt.show()

In [ ]:
adata.write_h5ad("/gpfs/Home/esm5360/bulk_rna_seq_analysis/data/combined_wt_ko_adata.h5ad")